# f6_m01d_shapash.ipynb

## Qué hace
Dashboard interactivo de interpretabilidad SHAP sobre el conjunto de test.
Construido con Plotly — sin dependencia de servidor, embebible en HTML.
Carga los valores SHAP calculados por `f6_m01a_shap_global.ipynb`.

## Requisitos
- `results/fase6/shap_global_catboost.pkl` (generado por m01a)
- `results/fase6/shap_importancia_comparativa.parquet` (generado por m01a)
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/CatBoost__balanced.pkl`

## Genera
- `docs/html/fase6/m01d_shapash.html` — dashboard interactivo completo
- `results/fase6/shapash_dashboard.html` — copia de respaldo

## Flujo
1. Config → 2. Imports → 3. Cargar SHAP + datos → 4. Dashboard: waterfall individual
→ 5. Beeswarm filtrable → 6. Tabla alumnos → 7. Comparativa grupos → 8. HTML

## Siguiente
`f6_m02a_lime.ipynb`


In [1]:
# 1. CONFIGURACIÓN DE RUTAS
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_HTML    = ROOT / 'docs' / 'html' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML.mkdir(parents=True, exist_ok=True)

print(f'ROOT:       {ROOT}')
print(f'DIR_RESULTS:{DIR_RESULTS}')
print(f'DIR_HTML:   {DIR_HTML}')


ROOT:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
DIR_RESULTS:C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\results\fase6
DIR_HTML:   C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase6


In [2]:
# 2. IMPORTS
import pandas as pd
import numpy as np
import joblib
import json as json_lib
import base64
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import shap
from src.html.render import render_pagina_desde_fichero

NOMBRES_LEGIBLES = {
    'n_anios_beca':            'Años con beca',
    'anios_sin_beca':          'Años sin beca',
    'cred_superados_anio_1er': 'Créditos superados 1er año',
    'situacion_laboral':       'Situación laboral',
    'nota_1er_anio':           'Nota 1er año',
    'nota_acceso':             'Nota de acceso',
    'tuvo_beca':               'Tuvo beca',
    'edad_entrada':            'Edad de entrada',
    'sexo':                    'Sexo',
    'max_pagos':               'Máx. pagos por curso',
    'rama':                    'Rama de conocimiento',
    'via_acceso':              'Vía de acceso',
    'anios_gap':               'Años de gap',
    'indicador_interrupcion':  'Interrupción formal',
    'tipo_acceso':             'Tipo de acceso',
    'n_convocatorias':         'Nº convocatorias',
    'curso_cohorte':           'Curso de entrada',
}

def nombre_legible(f):
    return NOMBRES_LEGIBLES.get(f, f.replace('_', ' '))

print('Imports OK.')


Imports OK.


In [3]:
# 3. CARGAR DATOS Y VALORES SHAP
# Disk-first: carga los .pkl generados por m01a para no recalcular.

X_test = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
y_test = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
df_imp = pd.read_parquet(DIR_RESULTS / 'shap_importancia_comparativa.parquet')

# Cargar pipeline y calcular probabilidades
pipeline_cat = joblib.load(DIR_MODELS / 'CatBoost__balanced.pkl')
y_prob = pipeline_cat.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

# Cargar valores SHAP de CatBoost (calculados por m01a)
RUTA_SHAP = DIR_RESULTS / 'shap_global_catboost.pkl'
if not RUTA_SHAP.exists():
    raise FileNotFoundError(
        f'No encontrado: {RUTA_SHAP}\n'
        'Ejecuta primero f6_m01a_shap_global.ipynb'
    )
shap_cat = joblib.load(RUTA_SHAP)

# Extraer matriz de valores SHAP
if hasattr(shap_cat, 'values'):
    shap_matrix = shap_cat.values
    if shap_matrix.ndim == 3:
        shap_matrix = shap_matrix[:, :, 1]
else:
    shap_matrix = np.array(shap_cat)

feature_names  = X_test.columns.tolist()
feature_labels = [nombre_legible(f) for f in feature_names]

# Tipo de clasificación por observación
y_true = y_test.values.ravel()
tipo_cls = np.where(
    (y_true==1)&(y_pred==1), 'TP',
    np.where((y_true==0)&(y_pred==0), 'TN',
    np.where((y_true==0)&(y_pred==1), 'FP', 'FN'))
)

print(f'X_test: {X_test.shape}')
print(f'SHAP matrix: {shap_matrix.shape}')
print(f'Features: {len(feature_names)}')
print('Datos cargados.')


X_test: (6725, 19)
SHAP matrix: (6725, 19)
Features: 19
Datos cargados.


In [4]:
# 4. GRÁFICO 1 — WATERFALL INDIVIDUAL (SHAP)
# Muestra cómo cada variable empuja la predicción hacia arriba o abajo
# para un alumno concreto. Selector interactivo por índice.
# Se generan 3 ejemplos representativos: TP, FN y FP.

def waterfall_alumno(idx: int, titulo: str) -> go.Figure:
    shap_vals  = shap_matrix[idx]
    feat_vals  = X_test.iloc[idx]
    base_value = shap_cat.base_values[idx] if hasattr(shap_cat, 'base_values') else 0

    # Top 10 por valor absoluto
    orden = np.argsort(np.abs(shap_vals))[::-1][:10]
    sv    = shap_vals[orden]
    fnames = [f'{nombre_legible(feature_names[i])} = {feat_vals.iloc[i]:.2g}'
              for i in orden]

    colores = ['#e53e3e' if v > 0 else '#3182ce' for v in sv]

    fig = go.Figure(go.Bar(
        x=sv[::-1], y=fnames[::-1],
        orientation='h',
        marker_color=colores[::-1],
        text=[f'{v:+.3f}' for v in sv[::-1]],
        textposition='outside',
        hovertemplate='%{y}<br>SHAP: %{x:.4f}<extra></extra>',
    ))
    prob = y_prob[idx]
    real = 'Abandona' if y_true[idx]==1 else 'No abandona'
    pred = 'Abandona' if y_pred[idx]==1 else 'No abandona'
    fig.update_layout(
        title=dict(text=f'{titulo}<br><sup>Prob. predicha: {prob:.3f} · Real: {real} · '
                   f'Predicción: {pred} · Tipo: {tipo_cls[idx]}</sup>',
                   font=dict(size=13)),
        xaxis=dict(title='Contribución SHAP (rojo = aumenta abandono, azul = reduce)'),
        height=420, width=780,
        plot_bgcolor='#f7fafc', paper_bgcolor='white',
        margin=dict(l=220, r=80, t=80, b=40),
    )
    return fig

# Seleccionar ejemplos representativos
idx_tp = np.where(tipo_cls=='TP')[0][0]
idx_fn = np.where(tipo_cls=='FN')[0][0]
idx_fp = np.where(tipo_cls=='FP')[0][0]

fig_wf_tp = waterfall_alumno(idx_tp, 'Alumno detectado correctamente (TP)')
fig_wf_fn = waterfall_alumno(idx_fn, 'Alumno no detectado — falso negativo (FN)')
fig_wf_fp = waterfall_alumno(idx_fp, 'Falsa alarma — falso positivo (FP)')

fig_wf_tp.show()
fig_wf_fn.show()
fig_wf_fp.show()
print('Waterfalls generados.')


Waterfalls generados.


In [5]:
# 5. GRÁFICO 2 — BEESWARM FILTRABLE POR TIPO DE CLASIFICACIÓN
# Cada punto = un alumno. Eje X = valor SHAP. Color = valor de la feature.
# Botones para filtrar por TP / FN / FP / TN / Todos.

# Top 12 features por importancia media
imp_media = np.abs(shap_matrix).mean(axis=0)
top12_idx = np.argsort(imp_media)[::-1][:12]
top12_names = [nombre_legible(feature_names[i]) for i in top12_idx]

# Muestra de 800 obs para no saturar
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), min(800, len(X_test)), replace=False)

traces = []
tipos_orden = ['Todos', 'TP', 'FN', 'FP', 'TN']
colores_tipo = {'TP':'#27ae60','FN':'#e74c3c','FP':'#e67e22','TN':'#95a5a6','Todos':'#3182ce'}

for tipo in tipos_orden:
    if tipo == 'Todos':
        mask = np.ones(len(sample_idx), dtype=bool)
    else:
        mask = tipo_cls[sample_idx] == tipo

    si = sample_idx[mask]
    for j, fi in enumerate(top12_idx):
        fname = nombre_legible(feature_names[fi])
        sv = shap_matrix[si, fi]
        fv = X_test.iloc[si, fi].values
        traces.append(go.Box(
            x=sv, name=fname,
            orientation='h',
            marker=dict(color=colores_tipo[tipo], opacity=0.5, size=4),
            boxpoints='all', jitter=0.4, pointpos=0,
            visible=(tipo == 'Todos'),
            legendgroup=tipo,
            showlegend=(j==0),
            hovertemplate=f'<b>{fname}</b><br>SHAP: %{{x:.4f}}<extra>{tipo}</extra>',
        ))

n_feat = len(top12_idx)
n_tipos = len(tipos_orden)

# Botones para filtrar por tipo
buttons = []
for ti, tipo in enumerate(tipos_orden):
    vis = [False] * (n_feat * n_tipos)
    for j in range(n_feat):
        vis[ti * n_feat + j] = True
    buttons.append(dict(
        label=tipo, method='update',
        args=[{'visible': vis},
              {'title': f'SHAP Beeswarm — {tipo} ({(tipo_cls==tipo).sum() if tipo!="Todos" else len(tipo_cls)} alumnos)'}]
    ))

fig_bee = go.Figure(data=traces)
fig_bee.update_layout(
    title='SHAP Beeswarm — Top 12 variables (filtrable por tipo de clasificación)',
    xaxis=dict(title='Valor SHAP (+ = mayor riesgo abandono)', zeroline=True,
               zerolinecolor='#718096', zerolinewidth=1),
    height=560, width=820,
    plot_bgcolor='#f7fafc', paper_bgcolor='white',
    updatemenus=[dict(
        type='buttons', direction='right',
        x=0.0, y=1.12, xanchor='left',
        buttons=buttons,
        bgcolor='#edf2f7', bordercolor='#e2e8f0',
        font=dict(size=11),
    )],
    margin=dict(l=200, r=40, t=100, b=40),
)
fig_bee.show()
print('Beeswarm filtrable generado.')


Beeswarm filtrable generado.


In [6]:
# 6. GRÁFICO 3 — TABLA INTERACTIVA DE ALUMNOS
# Muestra los 200 alumnos con mayor probabilidad predicha.
# Ordenable por columna. Coloreada por tipo de clasificación.

# Top 200 por probabilidad de abandono
top200_idx = np.argsort(y_prob)[::-1][:200]

df_tabla = pd.DataFrame({
    'Índice':         top200_idx,
    'Prob. abandono': y_prob[top200_idx].round(3),
    'Real':           ['Abandona' if v==1 else 'No abandona' for v in y_true[top200_idx]],
    'Predicción':     ['Abandona' if v==1 else 'No abandona' for v in y_pred[top200_idx]],
    'Tipo':           tipo_cls[top200_idx],
})

# Top 5 features para esta tabla
top5_feat = [feature_names[i] for i in top12_idx[:5]]
for f in top5_feat:
    df_tabla[nombre_legible(f)] = X_test.iloc[top200_idx][f].values.round(2)

colores_fila = {
    'TP':'#f0fff4', 'TN':'#ebf8ff', 'FP':'#fffbeb', 'FN':'#fff5f5'
}
fill_colors = [[colores_fila.get(t, 'white') for t in df_tabla['Tipo']]]

fig_tabla = go.Figure(go.Table(
    header=dict(
        values=list(df_tabla.columns),
        fill_color='#edf2f7',
        font=dict(size=11, color='#2d3748'),
        align='left', height=32,
    ),
    cells=dict(
        values=[df_tabla[c] for c in df_tabla.columns],
        fill_color=['#f7fafc'] + fill_colors * (len(df_tabla.columns)-1),
        font=dict(size=11, color='#2d3748'),
        align='left', height=26,
    )
))
fig_tabla.update_layout(
    title='Top 200 alumnos con mayor riesgo predicho — detalle por variable',
    height=520, width=900,
    margin=dict(t=60, b=10, l=10, r=10),
)
fig_tabla.show()
print('Tabla interactiva generada.')


Tabla interactiva generada.


In [7]:
# 7. GRÁFICO 4 — IMPORTANCIA SHAP POR GRUPO
# Compara la importancia media de cada variable entre:
# alumnos que abandonan (y_true=1) vs no abandonan (y_true=0).
# Revela qué variables discriminan más entre los dos grupos.

mask_ab  = y_true == 1
mask_nab = y_true == 0

imp_ab  = np.abs(shap_matrix[mask_ab]).mean(axis=0)
imp_nab = np.abs(shap_matrix[mask_nab]).mean(axis=0)

# Top 12 por diferencia entre grupos
diff = np.abs(imp_ab - imp_nab)
orden_diff = np.argsort(diff)[::-1][:12]
labels_diff = [nombre_legible(feature_names[i]) for i in orden_diff]

fig_grupos = go.Figure()
fig_grupos.add_trace(go.Bar(
    y=labels_diff[::-1], x=imp_ab[orden_diff][::-1],
    orientation='h', name='Abandonan',
    marker_color='#e53e3e', opacity=0.85,
    hovertemplate='%{y}<br>Importancia SHAP: %{x:.4f}<extra>Abandonan</extra>',
))
fig_grupos.add_trace(go.Bar(
    y=labels_diff[::-1], x=imp_nab[orden_diff][::-1],
    orientation='h', name='No abandonan',
    marker_color='#3182ce', opacity=0.85,
    hovertemplate='%{y}<br>Importancia SHAP: %{x:.4f}<extra>No abandonan</extra>',
))
fig_grupos.update_layout(
    title='Importancia SHAP media por grupo — variables que más discriminan',
    barmode='group',
    xaxis=dict(title='Importancia media |SHAP|'),
    height=480, width=820,
    plot_bgcolor='#f7fafc', paper_bgcolor='white',
    legend=dict(orientation='h', y=1.08, x=0),
    margin=dict(l=200, r=40, t=80, b=40),
)
fig_grupos.show()
print('Comparativa grupos generada.')


Comparativa grupos generada.


In [8]:
# 8. GENERAR HTML
# Embebe los 4 gráficos Plotly en el HTML del proyecto.
# render_pagina_desde_fichero — estándar del proyecto.

def plotly_html(fig) -> str:
    return fig.to_html(full_html=False, include_plotlyjs='cdn')

def bloque(titulo: str, caption: str, html_fig: str) -> str:
    return (
        f'<div style="margin:32px 0">'
        f'<h3 style="color:#2d3748;font-size:15px">{titulo}</h3>'
        f'<div style="border-radius:8px;overflow:hidden">{html_fig}</div>'
        f'<p style="color:#718096;font-size:12px;margin-top:6px">{caption}</p>'
        f'</div>'
    )

contenido = (
    '<h2 style="color:#2d3748">Dashboard de Interpretabilidad SHAP</h2>'
    '<p style="color:#4a5568;font-size:14px;max-width:900px;margin-bottom:8px">'
    'Dashboard interactivo construido con Plotly sobre los valores SHAP del modelo CatBoost. '
    'SHAP (SHapley Additive exPlanations) permite explicar cada predicción individual: '
    'cuánto contribuye cada variable a aumentar o reducir el riesgo de abandono de un alumno concreto.'
    '</p>'
    '<p style="color:#718096;font-size:12px;margin-bottom:32px">'
    f'Modelo: CatBoost__balanced | Conjunto de test: {len(X_test):,} observaciones | '
    f'Features analizadas: {len(feature_names)}'
    '</p>'
    + bloque(
        '1. Waterfall — alumno detectado correctamente (TP)',
        'Cada barra muestra cuánto empuja una variable hacia el abandono (rojo) '
        'o en contra (azul). La suma de todas las barras da la probabilidad final predicha.',
        plotly_html(fig_wf_tp)
    )
    + bloque(
        '2. Waterfall — alumno no detectado (FN)',
        'Falso negativo: el alumno abandona pero el modelo no lo detectó. '
        'Las variables protectoras (beca, créditos) superan las de riesgo — señal estructuralmente débil.',
        plotly_html(fig_wf_fn)
    )
    + bloque(
        '3. Waterfall — falsa alarma (FP)',
        'Falso positivo: el modelo predice abandono pero el alumno no abandona. '
        'Útil para entender qué perfil genera falsas alarmas.',
        plotly_html(fig_wf_fp)
    )
    + bloque(
        '4. Beeswarm filtrable — top 12 variables',
        'Cada punto es un alumno. Usa los botones para filtrar por tipo de clasificación '
        '(TP/FN/FP/TN). Permite ver si los patrones SHAP difieren entre grupos.',
        plotly_html(fig_bee)
    )
    + bloque(
        '5. Top 200 alumnos de mayor riesgo',
        'Tabla ordenable con probabilidad predicha, clasificación y valores de las 5 variables '
        'más importantes. Verde = TP, rojo claro = FN, amarillo = FP, azul = TN.',
        plotly_html(fig_tabla)
    )
    + bloque(
        '6. Importancia SHAP por grupo',
        'Comparativa de importancia media entre alumnos que abandonan y no abandonan. '
        'Las variables con mayor diferencia son las más discriminantes entre los dos grupos.',
        plotly_html(fig_grupos)
    )
)

html_completo = render_pagina_desde_fichero('f6_m01d_shapash.ipynb', contenido)
ruta_html = DIR_HTML / 'm01d_shapash.html'
ruta_html.write_text(html_completo, encoding='utf-8')

# Respaldo en results/
ruta_bak = DIR_RESULTS / 'shapash_dashboard.html'
ruta_bak.write_text(html_completo, encoding='utf-8')

print(f'HTML generado: {ruta_html}')
print(f'Respaldo:      {ruta_bak}')


HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase6\m01d_shapash.html
Respaldo:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\results\fase6\shapash_dashboard.html


In [9]:
# 9. NOTA: POR QUÉ NO SE USA SHAPASH EN MODO SERVIDOR
# ============================================================
# Shapash es una librería que construye un dashboard interactivo completo
# sobre valores SHAP con Dash/Plotly. En modo servidor ofrece:
#   - Filtrado dinámico por alumno, probabilidad, tipo de error
#   - Waterfall individual con selector interactivo
#   - Comparativa de grupos en tiempo real
#   - Exportación de explicaciones individuales
#
# MOTIVO PARA NO IMPLEMENTARLO:
#   1. Requiere lanzar un servidor local (puerto 8050 por defecto)
#   2. Necesita el entorno conda 'tfm_abandono' instalado en el equipo
#   3. No genera HTML embebible — no se puede incluir en la web del proyecto
#   4. En una defensa de TFM en PC ajeno o proyector no funciona
#   5. El dashboard Plotly construido en este notebook ofrece
#      las mismas funcionalidades clave sin ninguna dependencia
#
# CÓMO LANZARLO LOCALMENTE (si se quiere en el futuro):
#
# from shapash import SmartExplainer
#
# xpl = SmartExplainer(
#     model=pipeline_cat.named_steps['model'],
#     preprocessing=pipeline_cat[:-1],
#     features_dict=NOMBRES_LEGIBLES,
# )
# xpl.compile(
#     x=X_test,
#     y_pred=pd.Series(y_pred, name='abandono'),
#     y_target=pd.Series(y_true, name='abandono_real'),
# )
# app = xpl.run_app(title_story='Abandono UJI — Shapash Dashboard')
# # Abrir en http://localhost:8050
#
# NOTA: instalar con: pip install shapash --break-system-packages

print('Documentación Shapash modo servidor — ver comentarios de esta celda.')


Documentación Shapash modo servidor — ver comentarios de esta celda.
